 # 02_RLVR_Part2_Training_v45.ipynb



 ### 🤖 Dynamic Staggered-Sleeve PPO Alpha Agent Training

 - **Observation Space:** 46 Dimensions (12 CS Mean, 12 CS Std, 12 Benchmark Vector, 10 Stationary Macro)

 - **Action Space:** 16 Dimensions (12 Strategy Weights, Offset, Width, Equity Exposure, Active Tilt)

 - **MTM Portfolio:** Overlapping $H = \text{config.holding\_period}$ sleeves with T+1 Next-Day Close execution

 - **Objective:** Maximize benchmark-relative alpha spread under 100% equity exposure

In [1]:
# CELL 0: System Thread Locks, Cache Config & Reproducibility
import os
import torch

os.environ["CACHE_LOOKBACK"] = "41"
os.environ["CACHE_START_DATE"] = "1998-01-01"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
torch.set_num_threads(1)

In [2]:
# CELL 1: Start Execution Timer
import time

start_time = time.time()

In [3]:
# CELL 2: Environment & Path Setup
import sys
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2"
    )
else:
    rl_root_default = Path(
        r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2"
    )
    rl_root = rl_root_default if rl_root_default.exists() else Path.cwd()

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

Mounted at /content/drive


In [4]:
# AUTOMATED TRAINING GATE: Run before training starts
import pytest

ret_code = pytest.main(["-x", "tests/test_mtm_invariants.py", "-q"])
if ret_code != 0:
    raise SystemError(
        "❌ TRAINING ABORTED: MTM Invariant tripwires failed! Check tests/test_mtm_invariants.py"
    )
print("✅ ALL MTM INVARIANT TRIPWIRES PASSED. Safe to proceed with training.")

...                                                                      [100%]
3 passed in 20.25s
✅ ALL MTM INVARIANT TRIPWIRES PASSED. Safe to proceed with training.


In [5]:
# CELL 3: Imports
import copy
import gc
import json
import random
from typing import Any, Dict, Optional

import gymnasium as gym
import numpy as np
import pandas as pd

from core.paths import OUTPUT_DIR, LOCAL_DATA_DIR
from core.settings import CacheConfig, TradingConfig
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar
from rl_discovery.adapter import RLVRGymEnv
from rl_discovery.agent import AbsoluteZeroAgent
from rl_discovery.environment import DiscoveryEnv
from rl_discovery.trainer import PPOTrainer, RolloutBuffer
from rl_discovery.validator import AgentEvaluator

NOTEBOOKS_RLVR_ROOT: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2
PROJECT_ROOT: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks
GLOBAL_DATA_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data
GLOBAL_PROCESSED_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data/processed
LOCAL_DATA_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/data
OUTPUT_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output



In [6]:
# CELL 4: Load AlphaCache, Return Matrices & Split Timeline
print("📦 Loading preprocessed market data & AlphaCache...")
df_ohlcv, macro_df, features_df = load_processed_data()
config = TradingConfig()
df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)

cache_file_path = LOCAL_DATA_DIR / CacheConfig.get_filename()
print(f"AlphaCache path: {cache_file_path}")
feature_cube = pd.read_parquet(cache_file_path)

# Forward 1-Day MTM Simple Return Matrix (P_{t+1} - P_t) / P_t
simple_ret_matrix = df_close.pct_change(1, fill_method=None).shift(-1)
simple_ret_matrix["CASH"] = 0.0

# Split dates
valid_dates = feature_cube.index.get_level_values("Date").unique()
valid_calendar = trading_calendar[trading_calendar.isin(valid_dates)]

VAL_START = pd.Timestamp("2016-01-01")
TEST_START = pd.Timestamp("2022-04-01")

cal_train_raw = valid_calendar[valid_calendar < VAL_START]
cal_val_raw = valid_calendar[
    (valid_calendar >= VAL_START) & (valid_calendar < TEST_START)
]
cal_test_raw = valid_calendar[valid_calendar >= TEST_START]

# Purge leakage buffer based on config.holding_period
hp = config.holding_period
cal_train = cal_train_raw[:-hp] if len(cal_train_raw) > hp else cal_train_raw
cal_val = cal_val_raw[:-hp] if len(cal_val_raw) > hp else cal_val_raw
cal_test = cal_test_raw

print(f"✅ Data splits ready with {hp}-day leakage buffer dropped.")
print(
    f"   • Train: {cal_train.min().strftime('%Y-%m-%d')} to {cal_train.max().strftime('%Y-%m-%d')} ({len(cal_train)} days)"
)
print(
    f"   • Val  : {cal_val.min().strftime('%Y-%m-%d')} to {cal_val.max().strftime('%Y-%m-%d')} ({len(cal_val)} days)"
)
print(
    f"   • Test : {cal_test.min().strftime('%Y-%m-%d')} to {cal_test.max().strftime('%Y-%m-%d')} ({len(cal_test)} days)"
)

del df_ohlcv
gc.collect()

📦 Loading preprocessed market data & AlphaCache...
AlphaCache path: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/data/alpha_cache_41d_1998.parquet
✅ Data splits ready with 5-day leakage buffer dropped.
   • Train: 1998-01-02 to 2015-12-23 (4524 days)
   • Val  : 2016-01-04 to 2022-03-24 (1568 days)
   • Test : 2022-04-01 to 2026-08-19 (1099 days)


14

In [7]:
# CELL 5: Hyperparameter Grid Configuration
SLICE_NUM = 2  # Set to 1, 2, or 3 for distributed training runs

param_grid_sweep = [
    # ---------------- TAB 1: REWARD CONVEXITY & ACTIVE SPREAD FOCUS ----------------
    {
        "ID": "T1_Convex_2x_Scale20",
        "LR": 1.5e-4,
        "CRITIC_LR": 6.0e-4,
        "ENT_COEF_START": 0.0035,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 2.0,
        "loss_aversion_penalty": 0.0,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.70,
        "min_width": 3,
        "GAMMA": 0.90,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 20.0,
        "SEEDS": [42],
        "NUM_EPOCHS": 80,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 18,
        "NUM_STEPS": 1024,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 3,
        "WARMUP_EPOCHS": 8,
    },
    {
        "ID": "T1_Convex_3x_Scale20",
        "LR": 1.5e-4,
        "CRITIC_LR": 6.0e-4,
        "ENT_COEF_START": 0.0035,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 3.0,
        "loss_aversion_penalty": 0.0,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.70,
        "min_width": 3,
        "GAMMA": 0.90,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 20.0,
        "SEEDS": [42],
        "NUM_EPOCHS": 80,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 18,
        "NUM_STEPS": 1024,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 3,
        "WARMUP_EPOCHS": 8,
    },
    {
        "ID": "T1_Convex_25x_HighTilt85",
        "LR": 1.5e-4,
        "CRITIC_LR": 6.0e-4,
        "ENT_COEF_START": 0.0035,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 2.5,
        "loss_aversion_penalty": 0.0,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.85,
        "min_width": 3,
        "GAMMA": 0.90,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 20.0,
        "SEEDS": [42],
        "NUM_EPOCHS": 80,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 18,
        "NUM_STEPS": 1024,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 3,
        "WARMUP_EPOCHS": 8,
    },
    # ---------------- TAB 2: EXPLORATION & LEARNING RATE DYNAMICS ----------------
    {
        "ID": "T2_LR20_Ent005_Scale20",
        "LR": 2.0e-4,
        "CRITIC_LR": 8.0e-4,
        "ENT_COEF_START": 0.0050,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 2.0,
        "loss_aversion_penalty": 0.0,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.70,
        "min_width": 3,
        "GAMMA": 0.90,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 20.0,
        "SEEDS": [42],
        "NUM_EPOCHS": 80,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 18,
        "NUM_STEPS": 1024,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 3,
        "WARMUP_EPOCHS": 8,
    },
    {
        "ID": "T2_LR20_Ent005_Scale25",
        "LR": 2.0e-4,
        "CRITIC_LR": 8.0e-4,
        "ENT_COEF_START": 0.0050,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 2.5,
        "loss_aversion_penalty": 0.0,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.75,
        "min_width": 3,
        "GAMMA": 0.90,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 25.0,
        "SEEDS": [42],
        "NUM_EPOCHS": 80,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 18,
        "NUM_STEPS": 1024,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 3,
        "WARMUP_EPOCHS": 8,
    },
    {
        "ID": "T2_LR20_Ent006_Convex3x",
        "LR": 2.0e-4,
        "CRITIC_LR": 8.0e-4,
        "ENT_COEF_START": 0.0060,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 3.0,
        "loss_aversion_penalty": 0.0,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.70,
        "min_width": 3,
        "GAMMA": 0.90,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 20.0,
        "SEEDS": [42],
        "NUM_EPOCHS": 80,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 18,
        "NUM_STEPS": 1024,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 3,
        "WARMUP_EPOCHS": 8,
    },
    # ---------------- TAB 3: HORIZON & BASKET CONCENTRATION FOCUS ----------------
    {
        "ID": "T3_Gamma088_LR20_Scale25",
        "LR": 2.0e-4,
        "CRITIC_LR": 8.0e-4,
        "ENT_COEF_START": 0.0040,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 2.5,
        "loss_aversion_penalty": 0.0,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.80,
        "min_width": 3,
        "GAMMA": 0.88,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 25.0,
        "SEEDS": [42],
        "NUM_EPOCHS": 80,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 18,
        "NUM_STEPS": 1024,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 3,
        "WARMUP_EPOCHS": 8,
    },
    {
        "ID": "T3_Gamma088_Width5_Concentrated",
        "LR": 2.0e-4,
        "CRITIC_LR": 8.0e-4,
        "ENT_COEF_START": 0.0040,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 2.5,
        "loss_aversion_penalty": 0.0,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.80,
        "min_width": 5,
        "GAMMA": 0.88,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 25.0,
        "SEEDS": [42],
        "NUM_EPOCHS": 80,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 18,
        "NUM_STEPS": 1024,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 3,
        "WARMUP_EPOCHS": 8,
    },
    {
        "ID": "T3_Gamma090_Width5_Tilt85",
        "LR": 1.5e-4,
        "CRITIC_LR": 6.0e-4,
        "ENT_COEF_START": 0.0040,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 2.5,
        "loss_aversion_penalty": 0.0,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.85,
        "min_width": 5,
        "GAMMA": 0.90,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 20.0,
        "SEEDS": [42],
        "NUM_EPOCHS": 80,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 18,
        "NUM_STEPS": 1024,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 3,
        "WARMUP_EPOCHS": 8,
    },
]

slice_map = {
    1: param_grid_sweep[0:3],
    2: param_grid_sweep[3:6],
    3: param_grid_sweep[6:9],
}

if SLICE_NUM not in slice_map:
    raise ValueError(f"❌ SLICE_NUM must be 1, 2, or 3. Got: {SLICE_NUM}")

grid = slice_map[SLICE_NUM]
print("=" * 65)
print(
    f"🔥 ACTIVE TRAINING SESSION: SLICE {SLICE_NUM} of 3 ({len(grid)} Configurations)"
)
print("=" * 65)
for i, p in enumerate(grid, 1):
    print(
        f"  [{i}/{len(grid)}] ID: {p['ID']:<30} | LR: {p['LR']} | Mult: {p['UPSIDE_ALPHA_MULT']}x | Tilt: {p['min_active_tilt']}"
    )
print("=" * 65)

🔥 ACTIVE TRAINING SESSION: SLICE 2 of 3 (3 Configurations)
  [1/3] ID: T2_LR20_Ent005_Scale20         | LR: 0.0002 | Mult: 2.0x | Tilt: 0.7
  [2/3] ID: T2_LR20_Ent005_Scale25         | LR: 0.0002 | Mult: 2.5x | Tilt: 0.75
  [3/3] ID: T2_LR20_Ent006_Convex3x        | LR: 0.0002 | Mult: 3.0x | Tilt: 0.7


In [8]:
# CELL 6: Environment Factory
def create_fresh_environments(
    base_config: TradingConfig,
    grid_params: Dict[str, Any],
    df_close: pd.DataFrame,
    macro_df: pd.DataFrame,
    feature_cube: pd.DataFrame,
    cal_train: pd.DatetimeIndex,
    cal_val: pd.DatetimeIndex,
    cal_test: pd.DatetimeIndex,
):
    run_config = copy.deepcopy(base_config)

    if "loss_aversion_penalty" in grid_params:
        run_config.loss_aversion_penalty = float(grid_params["loss_aversion_penalty"])
    if "UPSIDE_ALPHA_MULT" in grid_params:
        run_config.upside_alpha_mult = float(grid_params["UPSIDE_ALPHA_MULT"])
    if "max_cash_pct" in grid_params:
        run_config.max_cash_pct = float(grid_params["max_cash_pct"])
    if "min_active_tilt" in grid_params:
        run_config.min_active_tilt = float(grid_params["min_active_tilt"])
    if "min_width" in grid_params:
        run_config.min_basket_width = int(grid_params["min_width"])
    elif "min_basket_width" in grid_params:
        run_config.min_basket_width = int(grid_params["min_basket_width"])

    simple_rets = df_close.pct_change(1, fill_method=None).shift(-1)
    simple_rets["CASH"] = 0.0

    def make_train_env():
        def thunk():
            e = DiscoveryEnv(
                feature_cube=feature_cube,
                simple_ret_matrix=simple_rets,
                calendar=cal_train,
                macro_df=macro_df,
                config=run_config,
                randomize_start=True,
                episode_steps=grid_params["NUM_STEPS"],
            )
            return RLVRGymEnv(e, macro_df)

        return thunk

    env_val = DiscoveryEnv(
        feature_cube=feature_cube,
        simple_ret_matrix=simple_rets,
        calendar=cal_val,
        macro_df=macro_df,
        config=run_config,
    )
    gym_val = RLVRGymEnv(env_val, macro_df)
    gym_val.is_training = False

    env_test = DiscoveryEnv(
        feature_cube=feature_cube,
        simple_ret_matrix=simple_rets,
        calendar=cal_test,
        macro_df=macro_df,
        config=run_config,
    )
    gym_test = RLVRGymEnv(env_test, macro_df)
    gym_test.is_training = False

    return make_train_env, gym_val, gym_test, run_config

In [9]:
# CELL 7: Checkpoint Cleaning & Single-Best Model Training Engine
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True, warn_only=True)


def get_hp_tag(params: Dict[str, Any], seed: int = 42) -> str:
    cid = params.get("ID", f"run_s{seed}")
    ent = params.get("ENT_COEF_START", 0.0035)
    mult = params.get("UPSIDE_ALPHA_MULT", 2.0)
    tilt = params.get("min_active_tilt", 0.70)
    w = params.get("min_width", params.get("min_basket_width", 3))
    gamma = params.get("GAMMA", 0.90)
    scale = params.get("REWARD_SCALE", 20.0)
    lr = params.get("LR", 1.5e-4)
    return (
        f"{cid}_s{seed}_ent{ent}_mult{mult}_tilt{tilt}_w{w}_g{gamma}_rsc{scale}_lr{lr}"
    )


def train_agent_for_grid(
    grid_params: Dict[str, Any],
    make_train_env,
    gym_val: RLVRGymEnv,
    checkpoint_dir: Path,
    run_config: Optional[TradingConfig] = None,
):
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    cfg = run_config or TradingConfig()

    seeds = grid_params["SEEDS"]
    num_epochs = grid_params["NUM_EPOCHS"]
    warmup_epochs = grid_params["WARMUP_EPOCHS"]
    num_envs = grid_params["NUM_ENVS"]

    grid_best_sharpe = -np.inf
    grid_best_model_path: Optional[Path] = None
    grid_best_seed = None
    grid_best_history = []
    grid_best_state_dict = None

    for seed in seeds:
        print(f"\n   -> 🚀 SEED: {seed}")
        set_seed(seed)
        hp_tag = get_hp_tag(grid_params, seed)

        # Remove any lingering stale checkpoints from prior aborted runs of this exact tag
        for existing_file in checkpoint_dir.glob(f"model_{hp_tag}_*.pt"):
            try:
                existing_file.unlink()
            except OSError:
                pass

        if "google.colab" in sys.modules:
            envs = gym.vector.AsyncVectorEnv(
                [make_train_env() for _ in range(num_envs)]
            )
        else:
            envs = gym.vector.SyncVectorEnv([make_train_env() for _ in range(num_envs)])

        obs_shape = envs.single_observation_space.shape
        action_shape = envs.single_action_space.shape
        assert obs_shape is not None and action_shape is not None

        obs_dim = int(obs_shape[0])
        action_dim = int(action_shape[0])

        agent = AbsoluteZeroAgent(obs_dim=obs_dim, action_dim=action_dim).to(device)
        trainer = PPOTrainer(
            agent,
            lr=grid_params["LR"],
            critic_lr=grid_params.get("CRITIC_LR", grid_params["LR"] * 4.0),
            clip_coef=0.2,
            clip_vloss=True,
            ent_coef=grid_params.get("ENT_COEF_START", 0.0035),
        )

        buffer = RolloutBuffer(
            num_steps=grid_params["NUM_STEPS"],
            num_envs=num_envs,
            obs_dim=obs_dim,
            action_dim=action_dim,
            device=device,
            gamma=grid_params.get("GAMMA", cfg.gamma),
            gae_lambda=grid_params.get("GAE_LAMBDA", cfg.gae_lambda),
        )

        seed_best_rolling_sharpe = -np.inf
        seed_current_checkpoint_path: Optional[Path] = None
        patience_counter = 0
        seed_history = []
        val_sharpe_history = []
        rolling_window = grid_params.get("ROLLING_EVAL_WINDOW", 3)

        for epoch in range(num_epochs):
            obs, _ = envs.reset(seed=seed) if epoch == 0 else envs.reset()
            epoch_rewards = []

            for step in range(grid_params["NUM_STEPS"]):
                obs_tensor = torch.tensor(obs, dtype=torch.float32).to(device)
                with torch.no_grad():
                    action, logprob, _, value = agent.get_action_and_value(obs_tensor)

                raw_action = action.cpu().numpy()
                next_obs, rewards, terminations, truncations, _ = envs.step(raw_action)

                if np.isnan(rewards).any() or np.isinf(rewards).any():
                    raise FloatingPointError(
                        f"NaN/Inf reward detected at Epoch {epoch+1}, Step {step}"
                    )

                reward_scale = grid_params.get("REWARD_SCALE", 20.0)
                scaled_rewards = rewards * reward_scale

                buffer.add(
                    obs=obs,
                    action=action,
                    logprob=logprob,
                    reward=scaled_rewards,
                    value=value,
                    terminations=terminations,
                    truncations=truncations,
                )
                epoch_rewards.append(rewards.mean())
                obs = next_obs

            # Bootstrap V(s_T)
            obs_tensor = torch.tensor(obs, dtype=torch.float32).to(device)
            next_term_tensor = torch.tensor(terminations, dtype=torch.float32).to(
                device
            )
            with torch.no_grad():
                next_values = agent.get_value(obs_tensor)

            buffer.compute_advantages(next_values, next_termination=next_term_tensor)

            trainer.update_schedules(
                epoch + 1,
                num_epochs,
                ent_start=grid_params.get("ENT_COEF_START", 0.0035),
                ent_end=grid_params.get("ENT_COEF_END", 0.0005),
            )

            diagnostics = trainer.update(
                buffer,
                update_epochs=grid_params["UPDATE_EPOCHS"],
                mini_batch_size=grid_params["MINI_BATCH_SIZE"],
            )
            buffer.step = 0

            diagnostics["epoch"] = epoch + 1
            diagnostics["avg_reward"] = float(np.mean(epoch_rewards))
            seed_history.append(diagnostics)

            if (epoch + 1) % grid_params["EVAL_FREQ"] == 0:
                train_scaler = envs.get_attr("scaler")[0]
                gym_val.scaler.load_state(train_scaler)

                val_results = AgentEvaluator.evaluate(agent, gym_val, device)
                val_sharpe = val_results["sharpe_ratio"]
                val_sharpe_history.append(val_sharpe)

                rolling_sharpe = float(np.mean(val_sharpe_history[-rolling_window:]))
                is_warmup = (epoch + 1) <= warmup_epochs
                status_icon = "🔥 [WARMUP]" if is_warmup else ""

                print(
                    f"      Ep {epoch+1:03d} | Rew: {diagnostics['avg_reward']:.4f} | "
                    f"Loss: {diagnostics['total_loss']:.4f} | EV: {diagnostics['explained_variance']:.3f} | "
                    f"Val Sharpe: {val_sharpe:.3f} | Rolling-{rolling_window}: {rolling_sharpe:.3f} {status_icon}"
                )

                if not is_warmup:
                    if rolling_sharpe > seed_best_rolling_sharpe:
                        seed_best_rolling_sharpe = rolling_sharpe
                        patience_counter = 0

                        new_checkpoint_name = f"model_{hp_tag}_ep_{epoch+1}_rollsh_{rolling_sharpe:.2f}_valsh_{val_sharpe:.2f}.pt"
                        new_checkpoint_path = checkpoint_dir / new_checkpoint_name

                        scaler_state = {
                            "mean": train_scaler.mean.copy(),
                            "var": train_scaler.var.copy(),
                            "count": train_scaler.count,
                        }

                        checkpoint_payload = {
                            "model_state_dict": agent.state_dict(),
                            "scaler_state": scaler_state,
                            "val_sharpe": val_sharpe,
                            "val_sortino": val_results["sortino_ratio"],
                            "val_excess_return": val_results["excess_return"],
                            "val_info_ratio": val_results["information_ratio"],
                            "rolling_val_sharpe": rolling_sharpe,
                            "epoch": epoch + 1,
                            "seed": seed,
                            "grid_params": grid_params,
                            "training_history": list(seed_history),
                        }

                        # Save new best model
                        torch.save(checkpoint_payload, str(new_checkpoint_path))

                        # Actively remove previous inferior model for this seed to ensure ONLY the best survives
                        if (
                            seed_current_checkpoint_path is not None
                            and seed_current_checkpoint_path.exists()
                        ):
                            try:
                                seed_current_checkpoint_path.unlink()
                            except OSError:
                                pass

                        seed_current_checkpoint_path = new_checkpoint_path
                        print(
                            f"         🟢 NEW BEST! Retained single best model (Rolling Sharpe: {rolling_sharpe:.3f})"
                        )

                        if rolling_sharpe > grid_best_sharpe:
                            grid_best_sharpe = rolling_sharpe
                            grid_best_model_path = new_checkpoint_path
                            grid_best_seed = seed
                            grid_best_history = list(seed_history)
                            grid_best_state_dict = copy.deepcopy(agent.state_dict())
                    else:
                        patience_counter += 1
                        print(
                            f"         🔴 No improvement. Patience: {patience_counter}/{grid_params['PATIENCE']}"
                        )

                    if patience_counter >= grid_params["PATIENCE"]:
                        print(f"      🛑 EARLY STOPPING triggered for Seed {seed}")
                        break

        envs.close()
        del agent, trainer, buffer, envs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return (
        grid_best_model_path,
        grid_best_sharpe,
        grid_best_seed,
        grid_best_history,
        grid_best_state_dict,
    )

In [10]:
# CELL 8: Grid Execution Loop
checkpoint_dir = OUTPUT_DIR / "model_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 60)
print(f"🏆 STARTING AUTOMATED HYPERPARAMETER SWEEP (SLICE {SLICE_NUM}) 🏆")
print("=" * 60)

for idx, params in enumerate(grid):
    current_seed = params["SEEDS"][0]
    hp_tag = get_hp_tag(params, current_seed)

    done_marker = checkpoint_dir / f".completed_{hp_tag}"
    if done_marker.exists():
        print(
            f"\n✅ [SKIPPING] GRID COMBO {idx+1}/{len(grid)} - Already completed: {hp_tag}"
        )
        continue

    print(f"\n\n{'='*50}")
    print(f"🔧 RUNNING GRID COMBO {idx+1}/{len(grid)}: {params['ID']}")
    print(json.dumps({k: v for k, v in params.items() if k != "SEEDS"}, indent=2))
    print(f"{'='*50}")

    make_train_env, gym_val, gym_test, run_config = create_fresh_environments(
        base_config=config,
        grid_params=params,
        df_close=df_close,
        macro_df=macro_df,
        feature_cube=feature_cube,
        cal_train=cal_train,
        cal_val=cal_val,
        cal_test=cal_test,
    )

    best_model_path, best_sharpe, best_seed, best_history, best_state_dict = (
        train_agent_for_grid(
            grid_params=params,
            make_train_env=make_train_env,
            gym_val=gym_val,
            checkpoint_dir=checkpoint_dir,
            run_config=run_config,
        )
    )

    done_marker.touch()

    if best_state_dict is None:
        print("⚠️ No model survived warmup for this configuration.")
        continue

    champion_name = best_model_path.name if best_model_path else "None"
    print(
        f"\n⭐ [GRID {idx+1} COMPLETE] Best Val Sharpe: {best_sharpe:.3f} | Best Model Saved: {champion_name}"
    )


🏆 STARTING AUTOMATED HYPERPARAMETER SWEEP (SLICE 2) 🏆


🔧 RUNNING GRID COMBO 1/3: T2_LR20_Ent005_Scale20
{
  "ID": "T2_LR20_Ent005_Scale20",
  "LR": 0.0002,
  "CRITIC_LR": 0.0008,
  "ENT_COEF_START": 0.005,
  "ENT_COEF_END": 0.0005,
  "UPSIDE_ALPHA_MULT": 2.0,
  "loss_aversion_penalty": 0.0,
  "max_cash_pct": 0.0,
  "min_active_tilt": 0.7,
  "min_width": 3,
  "GAMMA": 0.9,
  "GAE_LAMBDA": 0.95,
  "REWARD_SCALE": 20.0,
  "NUM_EPOCHS": 80,
  "UPDATE_EPOCHS": 4,
  "PATIENCE": 18,
  "NUM_STEPS": 1024,
  "MINI_BATCH_SIZE": 256,
  "NUM_ENVS": 8,
  "EVAL_FREQ": 2,
  "ROLLING_EVAL_WINDOW": 3,
  "WARMUP_EPOCHS": 8
}

   -> 🚀 SEED: 42
      Ep 002 | Rew: 0.0020 | Loss: -0.0504 | EV: -0.588 | Val Sharpe: 0.630 | Rolling-3: 0.630 🔥 [WARMUP]
      Ep 004 | Rew: 0.0018 | Loss: -0.0741 | EV: 0.374 | Val Sharpe: 0.714 | Rolling-3: 0.672 🔥 [WARMUP]
      Ep 006 | Rew: 0.0020 | Loss: -0.0705 | EV: 0.455 | Val Sharpe: 0.678 | Rolling-3: 0.674 🔥 [WARMUP]
      Ep 008 | Rew: 0.0020 | Loss: -0.0711 | EV: 0

In [11]:
# CELL 9: Execution Timing Summary
end_time = time.time()
print("\n" + "=" * 60)
print(
    f"🏆 TRAINING COMPLETED! Total Duration: {time.strftime('%H:%M:%S', time.gmtime(end_time - start_time))}"
)
print("Run Notebook 03a to evaluate the saved models Out-of-Sample.")
print("=" * 60)


🏆 TRAINING COMPLETED! Total Duration: 01:00:22
Run Notebook 03a to evaluate the saved models Out-of-Sample.


In [12]:
# CELL 10: Runtime Cleanup / Disconnect (Colab Safe)
if RUNNING_IN_COLAB:
    from google.colab import runtime  # type: ignore

    runtime.unassign()
    print("Runtime unassigned. Session disconnected.")

Runtime unassigned. Session disconnected.
